<a href="https://colab.research.google.com/github/salmaelhanchi/NeuroMatch_2026_Behaviour/blob/main/Data_Preperation_Notebook_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np


REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
REPO_DIR = Path("/content/NeuroMatch_2026_Behaviour")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

matches = list(REPO_DIR.rglob("data01_direction4priors.csv"))

if not matches:
    raise FileNotFoundError("data01_direction4priors.csv was not found inside the cloned repo.")

CSV_PATH = matches[0]
raw = pd.read_csv(CSV_PATH)

Cloning into '/content/NeuroMatch_2026_Behaviour'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 68 (delta 30), reused 38 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 3.53 MiB | 2.84 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [3]:
print(f"Loaded from: {CSV_PATH}")
print(f"raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

Loaded from: /content/NeuroMatch_2026_Behaviour/data01_direction4priors.csv
raw shape: 83,213 rows x 16 columns


,trial_index,trial_time,response_arrow_start_angle,motion_direction,motion_coherence,estimate_x,estimate_y,reaction_time,raw_response_time,prior_std,prior_mean,subject_id,experiment_name,experiment_id,session_id,run_id
0,1,0.000000,NaN,225,0.12,-1.749685,-1.785666,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
1,2,2.730730,NaN,225,0.12,-1.819693,-1.714269,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
2,3,4.913950,NaN,235,0.06,-1.562674,-1.951422,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
3,4,6.997296,NaN,225,0.06,-1.601388,-1.919781,NaN,NaN,10,225,1,data01_direction4priors,11,1,1
4,5,9.097130,NaN,215,0.24,-1.639461,-1.887371,NaN,NaN,10,225,1,data01_direction4priors,11,1,1


**Calculate Circular error estimation**

In [5]:
data = raw.copy()

# change repoted estimation in y and x coordinates to angles
data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360

#create circular estimation error
data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180
data[['motion_direction', 'estimate_deg', 'error_deg']].head()

,motion_direction,estimate_deg,error_deg
0,225,225.583113,0.583113
1,225,223.291282,-1.708718
2,235,231.312691,-3.687309
3,225,230.166776,5.166776
4,215,229.020860,14.020860


**Create trial-level table**

In [11]:
# create block_id for trial data
data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

# create trial-level table for 12 subjects
trial_data = data[['subject_id', 'block_id', 'trial_index', 'prior_std',
                    'motion_coherence', 'motion_direction',
                    'estimate_deg', 'error_deg']].copy()

print(trial_data.shape)
trial_data.head()

(83213, 8)


,subject_id,block_id,trial_index,prior_std,motion_coherence,motion_direction,estimate_deg,error_deg
0,1,1_1,1,10,0.12,225,225.583113,0.583113
1,1,1_1,2,10,0.12,225,223.291282,-1.708718
2,1,1_1,3,10,0.06,235,231.312691,-3.687309
3,1,1_1,4,10,0.06,225,230.166776,5.166776
4,1,1_1,5,10,0.24,215,229.020860,14.020860
